# RQ3: How does handling class imbalance improve fraud detection performance?

**Research Question:** Does addressing class imbalance (SMOTE, class weighting, undersampling) significantly improve recall and F1 for the minority fraud class?

**Hypothesis:** SMOTE oversampling will achieve the best recall for fraud cases compared to unbalanced and undersampled approaches.

**Target Variable:** `is_fraud`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
import warnings, os
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

print('Libraries loaded.')

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
DATA_PATH = '/kaggle/input/financial-transactions-dataset-for-fraud-detection/financial_fraud_detection_dataset.csv'
df = pd.read_csv(DATA_PATH, nrows=50000)

TARGET = 'is_fraud'
DROP_COLS = ['transaction_id','timestamp','sender_account','receiver_account',
             'fraud_type','ip_address','device_hash']
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)

le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col].astype(str))

df.dropna(inplace=True)
X = df.drop(columns=[TARGET])
y = df[TARGET]
print('Class distribution:', dict(y.value_counts()))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Define sampling strategies
sampling_strategies = {
    'No Resampling':    (X_train.copy(), y_train.copy()),
    'SMOTE':            SMOTE(random_state=42).fit_resample(X_train, y_train),
    'Undersampling':    RandomUnderSampler(random_state=42).fit_resample(X_train, y_train),
    'Class Weight':     (X_train.copy(), y_train.copy()),  # handled in model
}

results = []
for strategy, (Xtr, ytr) in sampling_strategies.items():
    cw = 'balanced' if strategy == 'Class Weight' else None
    model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight=cw)
    model.fit(Xtr, ytr)
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:,1]
    results.append({
        'Strategy':  strategy,
        'Train Size': len(ytr),
        'Accuracy':  round(accuracy_score(y_test, preds), 4),
        'Precision': round(precision_score(y_test, preds, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, preds, zero_division=0), 4),
        'F1':        round(f1_score(y_test, preds, zero_division=0), 4),
        'AUC-ROC':   round(roc_auc_score(y_test, proba), 4),
    })
    print(f'{strategy:20s} done.')

df_res = pd.DataFrame(results)
df_res

In [ ]:
df_res.to_csv('table_rq3_imbalance_handling.csv', index=False)
print('Table saved.')

In [ ]:
metrics = ['Precision', 'Recall', 'F1', 'AUC-ROC']
strategies = df_res['Strategy'].tolist()
x = np.arange(len(strategies))
width = 0.2
colors = ['#2196F3','#4CAF50','#FF9800','#E91E63']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: grouped bar chart
for i, (metric, color) in enumerate(zip(metrics, colors)):
    axes[0].bar(x + i*width, df_res[metric], width, label=metric, color=color, alpha=0.85, edgecolor='white')
axes[0].set_xticks(x + width*1.5)
axes[0].set_xticklabels(strategies, rotation=15, ha='right', fontsize=10)
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_ylim(0, 1.15)
axes[0].set_title('Metric Comparison by\nResampling Strategy', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].yaxis.grid(True, linestyle='--', alpha=0.6)
axes[0].set_axisbelow(True)

# Right: Recall vs Precision scatter
colors2 = ['#2196F3','#4CAF50','#FF9800','#E91E63']
for i, row in df_res.iterrows():
    axes[1].scatter(row['Precision'], row['Recall'], s=200, color=colors2[i],
                    label=row['Strategy'], zorder=5, edgecolors='black', linewidths=0.8)
    axes[1].annotate(row['Strategy'], (row['Precision'], row['Recall']),
                     textcoords='offset points', xytext=(6, 4), fontsize=9)
axes[1].set_xlabel('Precision', fontsize=12)
axes[1].set_ylabel('Recall', fontsize=12)
axes[1].set_title('Precision vs Recall\nTrade-off by Strategy', fontsize=12, fontweight='bold')
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.suptitle('RQ3: Impact of Class Imbalance Handling on Fraud Detection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figure_rq3_imbalance_handling.pdf', bbox_inches='tight', dpi=300)
plt.show()
print('Figure saved.')